## PROYECTO: PORTAL DE TICKETS PQRS
### ETAPA: CARGA (CAPA GOLD)
**OBJETIVO:** Generar agregaciones y KPIs listos para consumo analítico en herramientas de BI (ej. Power BI).

In [0]:
from pyspark.sql.functions import col, count, sum, when, current_timestamp

# Limpiamos widgets previos para evitar conflictos
dbutils.widgets.removeAll()

In [0]:
# 1. PARÁMETROS DINÁMICOS (WIDGETS)

dbutils.widgets.text("catalogo", "catalogo_pqrs")
dbutils.widgets.text("esquema_source", "silver")
dbutils.widgets.text("esquema_sink", "gold")

catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

print("✅ Parámetros cargados.")

In [0]:
# 2. LECTURA DE TABLA DESDE SILVER

df_silver = spark.table(f"{catalogo}.{esquema_source}.tickets_enriquecidos")

In [0]:
# 3. CREACIÓN DE DATAMART (AGREGACIONES Y KPIs)
# Agrupamos por Región y Nivel de Soporte para sacar conteos estadísticos.

from pyspark.sql.functions import count, sum, when, col, current_timestamp, expr

#df_gold = df_silver.groupBy("region_agente", "nivel_soporte").agg(
#    count(col("ticket_id")).alias("total_tickets"),
#    sum(when(col("criticidad") == "Alta", 1).otherwise(0)).alias("tickets_criticos")
#).orderBy(col("region_agente").desc(), col("total_tickets").desc())

spark.conf.set("spark.sql.ansi.enabled", "false")

df_gold = df_silver.groupBy("region_agente", "nivel_soporte").agg(
    count(col("ticket_id")).alias("total_tickets"),
    sum(col("criticidad")).alias("tickets_criticos")
).orderBy(col("region_agente").desc(), col("total_tickets").desc())

# Agregamos un timestamp para saber cuándo se calculó este KPI
df_gold = df_gold.withColumn("fecha_actualizacion_kpi", current_timestamp())

In [0]:
display(df_gold)

In [0]:
# 4. GUARDADO EN CAPA GOLD

tabla_destino = f"{catalogo}.{esquema_sink}.kpi_tickets_region"

df_gold.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_destino)

print(f"✅ Carga completada. Datamart analítico guardado en {tabla_destino}")